In [24]:
# === Section: Imports and Setup ===
import sys
from pathlib import Path
import logging
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
from typing import Set

# Find workspace root
current = Path().resolve()
while current.name != 'hku-datawork' and current.parent != current:
    current = current.parent
workspace_root = current if current.name == 'hku-datawork' else Path('/home/leo/GitHub_Clones/hku_stuff/hku-datawork')
sys.path.insert(0, str(workspace_root))

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Import modules
from hypothesis_testing.cointegration.data_loader import load_price_data
from hypothesis_testing.cointegration.data_split import split_data_chronologically
from hypothesis_testing.cointegration.basket_generator import generate_baskets_clustering
from hypothesis_testing.cointegration.utils_parallel import test_baskets_cointegration_parallel
from hypothesis_testing.cointegration.filter_sustainability import filter_baskets_sustainability
from hypothesis_testing.cointegration.filter_mean_reversion import filter_baskets_mean_reversion
from hypothesis_testing.cointegration.deduplicate_baskets import filter_overlapping_baskets
from hypothesis_testing.cointegration.hmm_regimes import train_and_persist_labels, apply_regime_filter

logger.info("Imports completed")

2025-11-07 15:55:07,360 - __main__ - INFO - Imports completed


# === Section: Configuration ===
# Set data paths, timeframes, and filter thresholds.

In [25]:
# === Configuration Parameters ===
DATA_DIR = Path("/workspace-hku/hku-data/test_data")  # Use absolute path to actual data location
TIMEFRAME = '15m'
PRICE_TYPE = 'mark'  # 15m only available as 'mark', 1h only as 'index'
SYMBOLS = None  # None = auto-discover all symbols

# Regime / cointegration tuning
COINTEGRATION_SAMPLE_SIZE = 2000  # Limit baskets for notebook runtime
REGIME_INCLUDE_LOW = {0}
REGIME_INCLUDE_HIGH = {1}
REGIME_POLICY = 'k_of_n'
REGIME_COVERAGE_THRESHOLD = 0.4
MAX_COMBINATIONS_PER_CLUSTER = 20000

ARTIFACTS_DIR = Path("hypothesis_testing/cointegration/artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True, parents=True)
REGIME_LABELS_PATH = ARTIFACTS_DIR / f"hmm_labels_{TIMEFRAME}_{PRICE_TYPE}.parquet"

# Discover available symbols from parquet files in data directory
pattern = f"*_{PRICE_TYPE}_{TIMEFRAME}_*.parquet"
parquet_files = list(DATA_DIR.glob(pattern))
available_symbols = set()
for f in parquet_files:
    # Pattern: perpetual_{SYMBOL}_mark_15m_...
    parts = f.stem.split('_')
    if len(parts) >= 2:
        symbol = parts[1]
        available_symbols.add(symbol)

AVAILABLE_SYMBOLS = sorted(list(available_symbols))
print(f"Found {len(AVAILABLE_SYMBOLS)} symbols available in {DATA_DIR}")

BARS_PER_DAY = 96


Found 116 symbols available in /workspace-hku/hku-data/test_data


# === Section: Load Price Data ===
# Load historical price data from parquet files.

In [26]:
# Load 15-minute mark price data from existing parquet files
# Data is already available in DATA_DIR - no download needed
logger.info(f"Loading data from parquet files in {DATA_DIR}")
logger.info(f"Timeframe: {TIMEFRAME}, Price type: {PRICE_TYPE}")

# Load data using the existing data loader (loads from parquet files)
# SYMBOLS=None means it will discover and load all symbols automatically
price_data = load_price_data(
    data_dir=DATA_DIR,
    years_back=1.2,  # Load only last 2 years
    symbols=None,  # Auto-discover all symbols from parquet files
    timeframe=TIMEFRAME,
    price_type=PRICE_TYPE,
    max_workers=None
)

print(f"  Symbols: {len(price_data.columns)}")
print(f"  Timestamps: {len(price_data):,}")
print(f"  Date range: {price_data.index.min()} to {price_data.index.max()}")


2025-11-07 15:55:07,404 - __main__ - INFO - Loading data from parquet files in /workspace-hku/hku-data/test_data
2025-11-07 15:55:07,407 - __main__ - INFO - Timeframe: 15m, Price type: mark
2025-11-07 15:55:07,411 - hypothesis_testing.cointegration.data_loader - INFO - Loading data from 116 parquet files for 116 symbols...
2025-11-07 15:55:08,757 - hypothesis_testing.cointegration.data_loader - INFO - Aligning timestamps across all symbols...
2025-11-07 15:55:09,577 - hypothesis_testing.cointegration.data_loader - INFO - Loaded 34939 timestamps for 104 symbols
2025-11-07 15:55:09,579 - hypothesis_testing.cointegration.data_loader - INFO - Date range: 2024-09-13 12:15:00+00:00 to 2025-09-12 10:45:00+00:00


  Symbols: 104
  Timestamps: 34,939
  Date range: 2024-09-13 12:15:00+00:00 to 2025-09-12 10:45:00+00:00


In [27]:
# Split data chronologically to prevent overfitting
splits = split_data_chronologically(
    price_data,
    cluster_split=(1.2, 0.8),      # Most recent 20% for clustering
    cointegration_split=(0.8, 0.2),
    test_split=(0.2, 0.0) 
)

cluster_data = splits['cluster_data']
cointegration_data = splits['cointegration_data']
test_data = splits['test_data']



2025-11-07 15:55:09,620 - hypothesis_testing.cointegration.data_split - INFO - Split data chronologically:
2025-11-07 15:55:09,621 - hypothesis_testing.cointegration.data_split - INFO -   Cluster: 6987 samples (1.2-80%) from 2025-07-01 16:15:00+00:00 to 2025-09-12 10:45:00+00:00
2025-11-07 15:55:09,623 - hypothesis_testing.cointegration.data_split - INFO -   Cointegration: 20964 samples (80%-20%) from 2024-11-25 07:15:00+00:00 to 2025-07-01 16:00:00+00:00
2025-11-07 15:55:09,624 - hypothesis_testing.cointegration.data_split - INFO -   Test: 6988 samples (20%-0%) from 2024-09-13 12:15:00+00:00 to 2024-11-25 07:00:00+00:00


In [28]:
# === Section: Train HMM Regime Labels ===
regime_meta = {
    "timeframe": TIMEFRAME,
    "price_type": PRICE_TYPE,
    "train_range": (
        cointegration_data.index.min().isoformat(),
        cointegration_data.index.max().isoformat(),
    ),
}
regime_labels = train_and_persist_labels(
    cointegration_data,
    BARS_PER_DAY,
    REGIME_LABELS_PATH,
    meta=regime_meta,
)
state_distribution = regime_labels.apply(pd.Series.value_counts).fillna(0)
state_share = state_distribution.sum(axis=1) / len(regime_labels)
print("Saved labels to:", REGIME_LABELS_PATH)
print("State share:")
print(state_share.sort_index())


Saved labels to: hypothesis_testing/cointegration/artifacts/hmm_labels_15m_mark.parquet
State share:
0    67.075588
1    36.924412
dtype: float64


In [29]:
# Generate candidate baskets (4-6 symbols) from cluster_data only
base_symbols = [col.replace('_close', '') for col in price_data.columns]

candidate_baskets = generate_baskets_clustering(
    price_data=cluster_data,
    n_clusters=10,
    max_combinations_per_cluster=MAX_COMBINATIONS_PER_CLUSTER,
)

logger.info(f"Generated {len(candidate_baskets)} candidate baskets")
print(f"Sample baskets: {candidate_baskets[:5]}")

2025-11-07 15:59:49,871 - hypothesis_testing.cointegration.basket_generator - INFO - Limited cluster size from 49 to 16 symbols (estimated 16122176 -> ~21760 combinations)
2025-11-07 15:59:49,904 - hypothesis_testing.cointegration.basket_generator - INFO - Limited cluster size from 32 to 16 symbols (estimated 1148984 -> ~21760 combinations)
2025-11-07 15:59:50,460 - __main__ - INFO - Generated 32255 candidate baskets


Sample baskets: [['PYTH', 'XTZ'], ['TON', 'MKR'], ['TON', 'BCH'], ['MKR', 'BCH'], ['TON', 'MKR', 'BCH']]


In [30]:
# === Section: Cointegration Testing (Baseline & Regimes) ===
if COINTEGRATION_SAMPLE_SIZE is not None:
    baskets_to_test = candidate_baskets[:COINTEGRATION_SAMPLE_SIZE]
else:
    baskets_to_test = candidate_baskets

tested_baskets = baskets_to_test

logger.info(
    "Testing %d baskets for cointegration at 1%% significance (deduplicated)",
    len(tested_baskets),
)

baseline_results = test_baskets_cointegration_parallel(
    price_data=cointegration_data,
    candidate_baskets=tested_baskets,
    max_workers=None,
    batch_size=100,
    deduplicate=True,
    overlap_threshold=0.5,
    prefer_lower_pvalue=True,
)

# Helper to measure raw regime coverage without fallback

def coverage_stats(include_states: Set[int]) -> np.ndarray:
    coverages = []
    for basket in tested_baskets:
        _, coverage = apply_regime_filter(
            price_data=cointegration_data,
            labels_df=regime_labels,
            basket=basket,
            include_states=include_states,
            policy=REGIME_POLICY,
            k=None,
            coverage_threshold=None,
        )
        coverages.append(coverage)
    return np.array(coverages)

low_coverage = coverage_stats(REGIME_INCLUDE_LOW)
high_coverage = coverage_stats(REGIME_INCLUDE_HIGH)

low_results = test_baskets_cointegration_parallel(
    price_data=cointegration_data,
    candidate_baskets=tested_baskets,
    regime_labels=regime_labels,
    include_states=REGIME_INCLUDE_LOW,
    policy=REGIME_POLICY,
    k=None,
    coverage_threshold=REGIME_COVERAGE_THRESHOLD,
    max_workers=None,
    batch_size=100,
    deduplicate=True,
    overlap_threshold=0.5,
    prefer_lower_pvalue=True,
)

high_results = test_baskets_cointegration_parallel(
    price_data=cointegration_data,
    candidate_baskets=tested_baskets,
    regime_labels=regime_labels,
    include_states=REGIME_INCLUDE_HIGH,
    policy=REGIME_POLICY,
    k=None,
    coverage_threshold=REGIME_COVERAGE_THRESHOLD,
    max_workers=None,
    batch_size=100,
    deduplicate=True,
    overlap_threshold=0.5,
    prefer_lower_pvalue=True,
)

cointegrated_baskets = baseline_results

print(f"Baseline cointegrated baskets: {len(baseline_results)}")
print(f"Low-vol cointegrated baskets: {len(low_results)}")
print(f"High-vol cointegrated baskets: {len(high_results)}")
print(f"Baseline hit-rate: {len(baseline_results)/len(tested_baskets):.1%}")
print(
    f"Low-vol coverage stats -> mean: {low_coverage.mean():.1%}, median: {np.median(low_coverage):.1%}"
)
print(
    f"High-vol coverage stats -> mean: {high_coverage.mean():.1%}, median: {np.median(high_coverage):.1%}"
)

if baseline_results:
    top_baseline = sorted(baseline_results, key=lambda b: b['johansen_result']['p_value'])[:5]
    print("Top baseline baskets (p-values):")
    for entry in top_baseline:
        print(f"  {entry['basket']} -> p={entry['johansen_result']['p_value']:.4f}")

if low_results:
    top_low = sorted(low_results, key=lambda b: b['johansen_result']['p_value'])[:5]
    print("Top low-vol baskets (p-values):")
    for entry in top_low:
        print(f"  {entry['basket']} -> p={entry['johansen_result']['p_value']:.4f}")

if high_results:
    top_high = sorted(high_results, key=lambda b: b['johansen_result']['p_value'])[:5]
    print("Top high-vol baskets (p-values):")
    for entry in top_high:
        print(f"  {entry['basket']} -> p={entry['johansen_result']['p_value']:.4f}")


2025-11-07 15:59:50,498 - __main__ - INFO - Testing 2000 baskets for cointegration at 1% significance (deduplicated)


2025-11-07 16:00:08,796 - hypothesis_testing.cointegration.utils_parallel - INFO - Processed 10/20 batches (1000 baskets), found 0 cointegrated so far
2025-11-07 16:00:17,696 - hypothesis_testing.cointegration.utils_parallel - INFO - Processed 20/20 batches (2000 baskets), found 0 cointegrated so far


TypeError: test_baskets_cointegration_parallel() got an unexpected keyword argument 'regime_labels'

In [ ]:
# Sustainability filter thresholds
PERSISTENCE_THRESHOLD = 0.7  # 70% of windows/periods must pass
WINDOW_DAYS = 30
STEP_DAYS = 10
PERIOD_DAYS = 30

sustainable_baskets = filter_baskets_sustainability(
    baskets=cointegrated_baskets,
    price_data=cointegration_data,  # Use cointegration_data
    persistence_threshold=PERSISTENCE_THRESHOLD,
    window_days=WINDOW_DAYS,
    step_days=STEP_DAYS,
    period_days=PERIOD_DAYS,
    bars_per_day=BARS_PER_DAY,
    use_rolling=True,
    use_discrete=True,
    max_workers=None
)

logger.info(
    f"Filtered to {len(sustainable_baskets)} sustainable baskets "
    f"(from {len(cointegrated_baskets)} cointegrated after dedup)"
)

# Display sample results
if sustainable_baskets:
    sample = sustainable_baskets[0]
    print(f"Sample sustainable basket: {sample['basket']}")
    print(
        f"Rolling windows persistence: "
        f"{sample['sustainability']['rolling_windows']['persistence_ratio']:.1%}"
    )
    print(
        f"Discrete periods persistence: "
        f"{sample['sustainability']['discrete_periods']['persistence_ratio']:.1%}"
    )


2025-11-07 13:24:38,125 - __main__ - INFO - Filtered to 0 sustainable baskets (from 0 cointegrated after dedup)


In [ ]:
HALF_LIFE_THRESHOLD_DAYS = 30
fast_mean_reverting_baskets = filter_baskets_mean_reversion(
    baskets=sustainable_baskets,
    test_data=test_data,  # Use test_data (prevents data leakage)
    half_life_threshold_days=HALF_LIFE_THRESHOLD_DAYS,
    bars_per_day=BARS_PER_DAY,
    max_workers=None
)

logger.info(f"Filtered to {len(fast_mean_reverting_baskets)} fast mean-reverting baskets "
           f"(from {len(sustainable_baskets)} sustainable)")

# Display sample results
if fast_mean_reverting_baskets:
    sample = fast_mean_reverting_baskets[0]
    print(f"\nSample fast mean-reverting basket: {sample['basket']}")
    print(f"Half-life: {sample['mean_reversion']['half_life_days']:.1f} days")
    print(f"ADF p-value: {sample['mean_reversion']['adf_p_value']:.4f}")
    print(f"Is stationary: {sample['mean_reversion']['is_stationary']}")

2025-11-06 13:51:21,002 - hypothesis_testing.cointegration.filter_mean_reversion - INFO - Testing 226 baskets for mean reversion speed (using separate test data: 13976 samples)...
2025-11-06 13:52:37,801 - hypothesis_testing.cointegration.filter_mean_reversion - INFO - Processed 10/226 baskets, found 0 fast mean-reverting so far
2025-11-06 13:52:52,844 - hypothesis_testing.cointegration.filter_mean_reversion - INFO - Processed 20/226 baskets, found 0 fast mean-reverting so far
2025-11-06 13:53:02,794 - hypothesis_testing.cointegration.filter_mean_reversion - INFO - Processed 30/226 baskets, found 0 fast mean-reverting so far
2025-11-06 13:53:58,186 - hypothesis_testing.cointegration.filter_mean_reversion - INFO - Processed 40/226 baskets, found 0 fast mean-reverting so far
2025-11-06 13:54:11,958 - hypothesis_testing.cointegration.filter_mean_reversion - INFO - Processed 50/226 baskets, found 0 fast mean-reverting so far
2025-11-06 13:54:24,703 - hypothesis_testing.cointegration.filter


Sample fast mean-reverting basket: ['GRT', 'NEAR', 'RENDER', 'ICP', 'FIL', 'SAND']
Half-life: inf days
ADF p-value: 0.0006
Is stationary: True


In [ ]:
import json

# Prepare baskets for saving (convert numpy arrays to lists)
validated_baskets_output = []
for basket_result in fast_mean_reverting_baskets:
    output = {
        'basket': basket_result['basket'],
        'eigenvector': basket_result['eigenvector'].tolist(),
        'johansen_p_value': float(basket_result['johansen_result']['p_value']),
        'johansen_trace_stat': float(basket_result['johansen_result']['trace_stat']),
        'sustainability_rolling': float(basket_result['sustainability']['rolling_windows']['persistence_ratio']),
        'sustainability_discrete': float(basket_result['sustainability']['discrete_periods']['persistence_ratio']),
        'half_life_days': float(basket_result['mean_reversion']['half_life_days']),
        'hurst_half_life_days': float(basket_result['mean_reversion']['half_life_days']),
    }
    validated_baskets_output.append(output)

# Save to JSON
output_file = Path("validated_baskets.json")
with open(output_file, 'w') as f:
    json.dump({
        'timestamp': datetime.now(timezone.utc).isoformat(),
        'config': {
            'timeframe': TIMEFRAME,
            'price_type': PRICE_TYPE,
            'persistence_threshold': PERSISTENCE_THRESHOLD,
            'half_life_threshold_days': HALF_LIFE_THRESHOLD_DAYS
        },
        'baskets': validated_baskets_output
    }, f, indent=2)

logger.info(f"Saved {len(validated_baskets_output)} validated baskets to {output_file}")

# Display summary
print(f"{'='*60}")
print("VALIDATION SUMMARY")
print(f"{'='*60}")
print(f"Total candidate baskets generated: {len(candidate_baskets)}")
print(f"Baskets tested in notebook: {len(tested_baskets)}")
cointegration_rate = (len(cointegrated_baskets) / len(tested_baskets)) if tested_baskets else 0.0
print(f"Cointegrated baskets (deduplicated, p<1%): {len(cointegrated_baskets)} ({cointegration_rate:.1%} of tested)")
if cointegrated_baskets:
    sustainable_rate = (len(sustainable_baskets) / len(cointegrated_baskets)) if cointegrated_baskets else 0.0
    print(f"Sustainable baskets: {len(sustainable_baskets)} ({sustainable_rate:.1%} of cointegrated)")
else:
    print("Sustainable baskets: 0")
if sustainable_baskets:
    fast_rate = (len(fast_mean_reverting_baskets) / len(sustainable_baskets)) if sustainable_baskets else 0.0
    print(f"Fast mean-reverting baskets: {len(fast_mean_reverting_baskets)} ({fast_rate:.1%} of sustainable)")
else:
    print("Fast mean-reverting baskets: 0")
print(f"{'='*60}")
print("Validated baskets ready for strategy deployment!")
print(f"Output file: {output_file}")


2025-11-06 14:07:21,954 - __main__ - INFO - Saved 2 validated baskets to validated_baskets.json


VALIDATION SUMMARY
Total candidate baskets: 7293
Cointegrated baskets (deduplicated): 226 (3.1%)
Sustainable baskets: 226 (100.0% of cointegrated)
Fast mean-reverting baskets: 2 (0.9% of sustainable)
Validated baskets ready for strategy deployment!
Output file: validated_baskets.json
